# 04 · Loss Function

Demonstrates why **MSE on raw RA/Dec is wrong** and motivates the
great-circle (angular separation) loss.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from src.training.loss import angular_separation_loss
from src.utils.coordinates import angular_separation_deg

## MSE wrap-around failure
Predict RA=359° when the truth is RA=1°. True separation is 2°, but
MSE thinks the error is 358°.

In [ ]:
ra_truth = 1.0
ra_preds = np.linspace(0, 360, 361)
mse = (ra_preds - ra_truth) ** 2
ang = angular_separation_deg(ra_preds, 0, ra_truth, 0)  # at equator
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(ra_preds, mse, color='#f56565', label='MSE (RA - RA_true)²')
ax2.plot(ra_preds, ang, color='#3ad29f', label='Great-circle separation (°)')
ax1.set_xlabel('Predicted RA (°)')
ax1.set_ylabel('MSE', color='#f56565'); ax2.set_ylabel('Great-circle Δ (°)', color='#3ad29f')
ax1.set_title('Truth RA=1°. MSE explodes near RA=180° but also wrongly at RA=359°.')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/04_mse_vs_angular.png', dpi=150)
plt.show()

## Loss landscape on the sphere

In [ ]:
ra_grid, dec_grid = np.meshgrid(np.linspace(0, 360, 181), np.linspace(-90, 90, 91))
loss = angular_separation_deg(ra_grid, dec_grid, 120, 30)
fig, ax = plt.subplots(figsize=(8, 4))
c = ax.pcolormesh(ra_grid, dec_grid, loss, cmap='viridis', shading='auto')
ax.scatter([120], [30], c='red', s=80, marker='*', label='True position')
fig.colorbar(c, ax=ax, label='Great-circle separation (°)')
ax.set_xlabel('RA (°)'); ax.set_ylabel('Dec (°)'); ax.legend()
ax.set_title('Loss landscape on the celestial sphere (no wrap discontinuity)')
fig.tight_layout(); fig.savefig(ROOT / 'reports/figures/04_loss_landscape.png', dpi=150)
plt.show()

## Torch loss sanity check

In [ ]:
pred = torch.tensor([[359.0, 0.0, 10.0, np.log(30.0)]])
target = torch.tensor([[1.0, 0.0, 10.0, np.log(30.0)]])
loss = angular_separation_loss(pred, target)
print(f'Loss for RA=359 vs RA=1 at equator: {loss.item():.6f} rad ({np.rad2deg(loss.item()):.3f}°)')
# Should be ~2° in radians = 0.0349, *not* 358° squared.